In [22]:
import os
import pandas as pd
import subprocess
from pathlib import Path
import pysrt

run python .\trimming.py -original .\fbx\palmer\unprocessed\M20240925_1858_260319_0.fbx -post .\fbx\palmer\processed\M20240925_1858_260319_0.fbx -bone head upperarm_l upperarm_r for every unique fbx filename

In [23]:
# Get all filenames in the unprocessed directory
unprocessed_dir = r'..\fbx\palmer\unprocessed'
processed_dir = r'..\fbx\palmer\processed'

unprocessed_files = [f for f in os.listdir(unprocessed_dir) if f.endswith('.fbx')]
processed_files = [f for f in os.listdir(processed_dir) if f.endswith('.fbx')]

In [24]:
# Run the command to get all start and end offsets
# Read info from output/trimming.csv as originalName,postName,startOffset,endOffset,error
trimming_csv = Path('output') / 'trimming.csv'
columns = ['originalName', 'postName', 'startOffset', 'endOffset', 'error']

if trimming_csv.exists():
    trimming_csv.unlink()

trimming_list = []

for i, unprocessed_file in enumerate(unprocessed_files):
    # Find the corresponding processed file
    processed_file = next((f for f in processed_files if f.startswith(unprocessed_file[:-4])), None)
    if not processed_file:
        print(f'No corresponding processed file found for {unprocessed_file}')
        continue

    # Run the trimming.py script with the appropriate arguments
    original_path = Path(unprocessed_dir) / unprocessed_file
    post_path = Path(processed_dir) / processed_file
    command = [
        'python', 'trimming.py',
        '-original', str(original_path),
        '-post', str(post_path),
        '-bone', 'head', 'upperarm_l', 'upperarm_r'
    ]
    subprocess.run(command)

    # cur_csv = pd.read_csv(trimming_csv, header=None, names=columns)
    # if not cur_csv.empty:
    #     trimming_list.append(cur_csv)

    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1} files out of {len(unprocessed_files)}')

# print(len(trimming_list), 'trimming results collected')


Processed 10 files out of 400
Processed 20 files out of 400
Processed 30 files out of 400
Processed 40 files out of 400
Processed 50 files out of 400
Processed 60 files out of 400
Processed 70 files out of 400
Processed 80 files out of 400
Processed 90 files out of 400
Processed 100 files out of 400
Processed 110 files out of 400
Processed 120 files out of 400
Processed 130 files out of 400
Processed 140 files out of 400
Processed 150 files out of 400
Processed 160 files out of 400
Processed 170 files out of 400
Processed 180 files out of 400
Processed 190 files out of 400
Processed 200 files out of 400
Processed 210 files out of 400
Processed 220 files out of 400
Processed 230 files out of 400
Processed 240 files out of 400
Processed 250 files out of 400
Processed 260 files out of 400
Processed 270 files out of 400
Processed 280 files out of 400
Processed 290 files out of 400
Processed 300 files out of 400
Processed 310 files out of 400
Processed 320 files out of 400
Processed 330 fil

In [27]:
# trimming_df = pd.concat(trimming_list, ignore_index=True) if trimming_list else pd.DataFrame(columns=columns)

trimming_df = pd.read_csv(trimming_csv, header=None, names=columns) if trimming_csv.exists() else pd.DataFrame(columns=columns)

print(len(trimming_df), 'rows in trimming_df')

trimming_df.head()

400 rows in trimming_df


,originalName,postName,startOffset,endOffset,error
0,M20240925_1858_260319_0.fbx,M20240925_1858_260319_0.fbx,23,0,0.129
1,M20240925_1872_260318_0.fbx,M20240925_1872_260318_0.fbx,0,0,0.837
2,M20240925_2050_260319_1.fbx,M20240925_2050_260319_1.fbx,0,8,0.031
3,M20241106_5930_260319_2.fbx,M20241106_5930_260319_2.fbx,19,9,0.044
4,M20241106_5991_260318_5.fbx,M20241106_5991_260318_5.fbx,3,15,0.135


In [28]:
# For each entry, convert the srt from 30 to 24 fps (so seconds * (30/24), so * 1.25), then add the startOffset to the timecodes
srt_dir = r'..\..\AnimData\srt'
save_dir = r'..\..\AnimData\srt_fixed'

#srt name in form of {NAME}_Signbank_ID_glossen.srt

for index, row in trimming_df.iterrows():
    name = row['originalName'][:-4]  # Remove .fbx
    srt_path = Path(srt_dir) / f'{name}_Signbank_ID_glossen.srt'
    save_path = Path(save_dir) / f'{name}_Signbank_ID_glossen.srt'

    srt = pysrt.open(srt_path)

    # Covert frames to seconds (startOffset is in 24 fps, so /24)
    startOffset_seconds = row['startOffset'] / 24

    # Convert to 24 fps and add startOffset
    for sub in srt:
        sub.start = pysrt.SubRipTime.from_ordinal(int(sub.start.ordinal * (30 / 24) + startOffset_seconds * 1000))
        sub.end = pysrt.SubRipTime.from_ordinal(int(sub.end.ordinal * (30 / 24) + startOffset_seconds * 1000))

    srt.save(save_path)

    if (index + 1) % 10 == 0:
        print(f'Processed {index + 1} SRT files out of {len(trimming_df)}')

Processed 10 SRT files out of 400
Processed 20 SRT files out of 400
Processed 30 SRT files out of 400
Processed 40 SRT files out of 400
Processed 50 SRT files out of 400
Processed 60 SRT files out of 400
Processed 70 SRT files out of 400
Processed 80 SRT files out of 400
Processed 90 SRT files out of 400
Processed 100 SRT files out of 400
Processed 110 SRT files out of 400
Processed 120 SRT files out of 400
Processed 130 SRT files out of 400
Processed 140 SRT files out of 400
Processed 150 SRT files out of 400
Processed 160 SRT files out of 400
Processed 170 SRT files out of 400
Processed 180 SRT files out of 400
Processed 190 SRT files out of 400
Processed 200 SRT files out of 400
Processed 210 SRT files out of 400
Processed 220 SRT files out of 400
Processed 230 SRT files out of 400
Processed 240 SRT files out of 400
Processed 250 SRT files out of 400
Processed 260 SRT files out of 400
Processed 270 SRT files out of 400
Processed 280 SRT files out of 400
Processed 290 SRT files out o